In [4]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
REPORTS_DIR = Path(
    r"C:\Users\khush\Desktop\IDX-Exchange\Reports"
)

MAP_DIR = REPORTS_DIR / "Map_Checks"
MAP_DIR.mkdir(parents=True, exist_ok=True)

# Load FINAL Week 7 dataset
sold = pd.read_csv(
    REPORTS_DIR / "sold_filtered.csv",
    low_memory=False
)

print(f"Dataset loaded: {len(sold):,} rows x {sold.shape[1]} columns")


# -----------------------------
# Make coordinates numeric
# -----------------------------
sold["Latitude"] = pd.to_numeric(
    sold["Latitude"],
    errors="coerce"
)

sold["Longitude"] = pd.to_numeric(
    sold["Longitude"],
    errors="coerce"
)

map_data = sold.dropna(
    subset=["Latitude", "Longitude"]
).copy()

print(f"Rows with coordinates: {len(map_data):,}")
print(f"Rows missing coordinates: {len(sold) - len(map_data):,}")

Dataset loaded: 437,455 rows x 93 columns
Rows with coordinates: 433,208
Rows missing coordinates: 4,247


In [5]:
# -----------------------------
# California bounding box
# -----------------------------
CA_MIN_LAT = 32
CA_MAX_LAT = 42
CA_MIN_LON = -125
CA_MAX_LON = -114

map_data["outside_ca_check"] = ~(
    map_data["Latitude"].between(CA_MIN_LAT, CA_MAX_LAT)
    &
    map_data["Longitude"].between(CA_MIN_LON, CA_MAX_LON)
)

outside_ca = map_data[
    map_data["outside_ca_check"]
].copy()

print("=== CALIFORNIA BOUNDS CHECK ===")
print(f"Properties checked: {len(map_data):,}")
print(f"Outside CA bounds: {len(outside_ca):,}")

review_cols = [
    "ListingKey",
    "City",
    "CountyOrParish",
    "StateOrProvince",
    "StateForAnalysis",
    "PostalCode",
    "PostalCode5",
    "Latitude",
    "Longitude"
]

review_cols = [
    col for col in review_cols
    if col in outside_ca.columns
]

display(outside_ca[review_cols])

=== CALIFORNIA BOUNDS CHECK ===
Properties checked: 433,208
Outside CA bounds: 0


,ListingKey,City,CountyOrParish,StateOrProvince,StateForAnalysis,PostalCode,PostalCode5,Latitude,Longitude


In [6]:
# -----------------------------
# MAP 1: Full property heat map
# -----------------------------
full_map = folium.Map(
    location=[36.5, -119.5],
    zoom_start=6,
    tiles="CartoDB positron"
)

heat_data = map_data[
    ["Latitude", "Longitude"]
].values.tolist()

HeatMap(
    heat_data,
    radius=6,
    blur=8,
    min_opacity=0.25,
    max_zoom=10
).add_to(full_map)

full_map_path = MAP_DIR / "01_full_california_heatmap.html"

full_map.save(full_map_path)

print("Saved:")
print(full_map_path)

Saved:
C:\Users\khush\Desktop\IDX-Exchange\Reports\Map_Checks\01_full_california_heatmap.html


In [7]:
# -----------------------------
# MAP 2: Outside California
# -----------------------------
outside_map = folium.Map(
    location=[36.5, -119.5],
    zoom_start=5,
    tiles="CartoDB positron"
)

for _, row in outside_ca.iterrows():

    popup = f"""
    <b>ListingKey:</b> {row.get('ListingKey', '')}<br>
    <b>City:</b> {row.get('City', '')}<br>
    <b>County:</b> {row.get('CountyOrParish', '')}<br>
    <b>Original State:</b> {row.get('StateOrProvince', '')}<br>
    <b>Analysis State:</b> {row.get('StateForAnalysis', '')}<br>
    <b>ZIP:</b> {row.get('PostalCode', '')}<br>
    <b>ZIP5:</b> {row.get('PostalCode5', '')}<br>
    <b>Latitude:</b> {row['Latitude']}<br>
    <b>Longitude:</b> {row['Longitude']}
    """

    folium.CircleMarker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],
        radius=6,
        popup=folium.Popup(
            popup,
            max_width=350
        ),
        fill=True,
        fill_opacity=0.8
    ).add_to(outside_map)

outside_map_path = MAP_DIR / "02_outside_california.html"

outside_map.save(outside_map_path)

print(f"Outside-CA properties mapped: {len(outside_ca):,}")
print("Saved:")
print(outside_map_path)

Outside-CA properties mapped: 0
Saved:
C:\Users\khush\Desktop\IDX-Exchange\Reports\Map_Checks\02_outside_california.html


In [8]:
# -----------------------------
# MAP 3: State/coordinate mismatches
# -----------------------------
if "state_coordinate_mismatch_flag" in map_data.columns:

    mismatch = map_data[
        map_data["state_coordinate_mismatch_flag"] == True
    ].copy()

else:

    # Recalculate it independently
    state = (
        map_data["StateOrProvince"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    inside_ca = (
        map_data["Latitude"].between(CA_MIN_LAT, CA_MAX_LAT)
        &
        map_data["Longitude"].between(CA_MIN_LON, CA_MAX_LON)
    )

    mismatch = map_data[
        inside_ca
        &
        state.notna()
        &
        ~state.isin(["CA", "CALIFORNIA", ""])
    ].copy()


print(f"State/coordinate mismatches: {len(mismatch):,}")


mismatch_map = folium.Map(
    location=[36.5, -119.5],
    zoom_start=6,
    tiles="CartoDB positron"
)


for _, row in mismatch.iterrows():

    popup = f"""
    <b>ListingKey:</b> {row.get('ListingKey', '')}<br>
    <b>City:</b> {row.get('City', '')}<br>
    <b>County:</b> {row.get('CountyOrParish', '')}<br>
    <b>MLS State:</b> {row.get('StateOrProvince', '')}<br>
    <b>StateForAnalysis:</b> {row.get('StateForAnalysis', '')}<br>
    <b>ZIP:</b> {row.get('PostalCode', '')}<br>
    <b>Latitude:</b> {row['Latitude']}<br>
    <b>Longitude:</b> {row['Longitude']}
    """

    folium.CircleMarker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],
        radius=6,
        popup=folium.Popup(
            popup,
            max_width=350
        ),
        fill=True,
        fill_opacity=0.8
    ).add_to(mismatch_map)


mismatch_map_path = (
    MAP_DIR / "03_state_coordinate_mismatches.html"
)

mismatch_map.save(mismatch_map_path)

print("Saved:")
print(mismatch_map_path)

State/coordinate mismatches: 13
Saved:
C:\Users\khush\Desktop\IDX-Exchange\Reports\Map_Checks\03_state_coordinate_mismatches.html


In [9]:
# -----------------------------
# MAP 4: Malformed ZIP records
# -----------------------------
if "malformed_postal_code_flag" in map_data.columns:

    bad_zip = map_data[
        map_data["malformed_postal_code_flag"] == True
    ].copy()

else:

    postal = (
        map_data["PostalCode"]
        .astype("string")
        .str.strip()
    )

    valid_zip = postal.str.match(
        r"^\d{5}(?:-\d{4})?$",
        na=False
    )

    bad_zip = map_data[
        postal.notna()
        &
        ~valid_zip
    ].copy()


print(f"Malformed ZIP records with coordinates: {len(bad_zip):,}")


bad_zip_map = folium.Map(
    location=[36.5, -119.5],
    zoom_start=6,
    tiles="CartoDB positron"
)


for _, row in bad_zip.iterrows():

    popup = f"""
    <b>ListingKey:</b> {row.get('ListingKey', '')}<br>
    <b>City:</b> {row.get('City', '')}<br>
    <b>County:</b> {row.get('CountyOrParish', '')}<br>
    <b>State:</b> {row.get('StateOrProvince', '')}<br>
    <b>Original ZIP:</b> {row.get('PostalCode', '')}<br>
    <b>Clean ZIP:</b> {row.get('PostalCode5', '')}<br>
    <b>Latitude:</b> {row['Latitude']}<br>
    <b>Longitude:</b> {row['Longitude']}
    """

    folium.CircleMarker(
        location=[
            row["Latitude"],
            row["Longitude"]
        ],
        radius=6,
        popup=folium.Popup(
            popup,
            max_width=350
        ),
        fill=True,
        fill_opacity=0.8
    ).add_to(bad_zip_map)


bad_zip_map_path = (
    MAP_DIR / "04_malformed_zip_locations.html"
)

bad_zip_map.save(bad_zip_map_path)

print("Saved:")
print(bad_zip_map_path)

Malformed ZIP records with coordinates: 32
Saved:
C:\Users\khush\Desktop\IDX-Exchange\Reports\Map_Checks\04_malformed_zip_locations.html
